# Week 2 - Day 5: Complete Exploratory Data Analysis

This notebook combines the Week 2 EDA concepts into a single narrated analysis of the music dataset.

## 1. Objective

The goal is to inspect the dataset, check its quality, explore the distributions of important features, examine relationships between variables, and summarize the findings for future modeling.

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Resolve the dataset from different possible working directories.
possible_paths = []
for base in [Path.cwd(), *Path.cwd().parents[:3]]:
    possible_paths.extend([
        base / 'data.csv',
        base / 'week2/day4/data.csv',
        base / 'week2/day4/../day4/data.csv',
    ])

for candidate in possible_paths:
    if candidate.exists():
        data_path = candidate
        break
else:
    raise FileNotFoundError('Could not find the music dataset.')

df = pd.read_csv(data_path)

if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

print('Dataset shape:', df.shape)
df.head()


Dataset shape: (2017, 16)


,acousticness,danceability,duration_ms,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,time_signature,valence,target,song_title,artist
0,0.0102,0.833,204600,0.434,0.021900,2,0.1650,-8.795,1,0.4310,150.062,4.0,0.286,1,Mask Off,Future
1,0.1990,0.743,326933,0.359,0.006110,1,0.1370,-10.401,1,0.0794,160.083,4.0,0.588,1,Redbone,Childish Gambino
2,0.0344,0.838,185707,0.412,0.000234,2,0.1590,-7.148,1,0.2890,75.044,4.0,0.173,1,Xanny Family,Future
3,0.6040,0.494,199413,0.338,0.510000,5,0.0922,-15.236,1,0.0261,86.468,4.0,0.230,1,Master Of None,Beach House
4,0.1800,0.678,392893,0.561,0.512000,5,0.4390,-11.648,0,0.0694,174.004,4.0,0.904,1,Parallel Lines,Junior Boys


## 2. Dataset overview

In [2]:
print(df.shape)
df.info()
print(df.dtypes)

(2017, 16)
<class 'pandas.DataFrame'>
RangeIndex: 2017 entries, 0 to 2016
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   acousticness      2017 non-null   float64
 1   danceability      2017 non-null   float64
 2   duration_ms       2017 non-null   int64  
 3   energy            2017 non-null   float64
 4   instrumentalness  2017 non-null   float64
 5   key               2017 non-null   int64  
 6   liveness          2017 non-null   float64
 7   loudness          2017 non-null   float64
 8   mode              2017 non-null   int64  
 9   speechiness       2017 non-null   float64
 10  tempo             2017 non-null   float64
 11  time_signature    2017 non-null   float64
 12  valence           2017 non-null   float64
 13  target            2017 non-null   int64  
 14  song_title        2017 non-null   str    
 15  artist            2017 non-null   str    
dtypes: float64(10), int64(4), str(2)
memory us

## 3. Missing values and duplicate detection

In [3]:
print('Missing values:')
print(df.isnull().sum())

duplicate_count = df.duplicated().sum()
print('Duplicate rows before cleaning:', duplicate_count)

if duplicate_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)

print('Duplicate rows after cleaning:', df.duplicated().sum())
print('Dataset shape after cleaning:', df.shape)

Missing values:
acousticness        0
danceability        0
duration_ms         0
energy              0
instrumentalness    0
key                 0
liveness            0
loudness            0
mode                0
speechiness         0
tempo               0
time_signature      0
valence             0
target              0
song_title          0
artist              0
dtype: int64
Duplicate rows before cleaning: 5
Duplicate rows after cleaning: 0
Dataset shape after cleaning: (2012, 16)


The dataset had no missing values, and 5 duplicated rows were removed before the main analysis. The cleaned dataset was then used for the rest of the notebook.

## 4. Descriptive statistics

In [4]:
numeric_df = df.select_dtypes(include=np.number)
print(numeric_df.describe())

       acousticness  danceability   duration_ms       energy  \
count   2012.000000   2012.000000  2.012000e+03  2012.000000   
mean       0.187513      0.618450  2.462608e+05     0.681840   
std        0.259691      0.161003  8.202146e+04     0.210255   
min        0.000003      0.122000  1.604200e+04     0.014800   
25%        0.009590      0.514000  2.000045e+05     0.563750   
50%        0.063500      0.631000  2.291200e+05     0.715500   
75%        0.265000      0.738000  2.703565e+05     0.846000   
max        0.995000      0.984000  1.004627e+06     0.998000   

       instrumentalness          key     liveness     loudness         mode  \
count       2012.000000  2012.000000  2012.000000  2012.000000  2012.000000   
mean           0.132980     5.348907     0.190816    -7.076750     0.612326   
std            0.272967     3.649559     0.155571     3.756502     0.487341   
min            0.000000     0.000000     0.018800   -33.097000     0.000000   
25%            0.000000     

## 5. Target-class distribution

In [5]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='target')
plt.title('Target Class Distribution')
plt.xlabel('Target Class')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

print(df['target'].value_counts())

target
1    1015
0     997
Name: count, dtype: int64


/tmp/ipykernel_599619/1164339183.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The two classes are close to balanced, so the dataset does not appear strongly skewed toward one label.

## 6. Univariate numeric analysis

In [6]:
selected_features = ['danceability', 'energy', 'loudness', 'tempo']
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, feature in zip(axes.flatten(), selected_features):
    df[feature].hist(ax=ax, bins=20, edgecolor='black')
    ax.set_title(feature.capitalize())
    ax.set_xlabel(feature.capitalize())
    ax.set_ylabel('Frequency')
plt.tight_layout()
plt.show()

/tmp/ipykernel_599619/3363583337.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The histograms show that danceability and tempo are fairly spread out, while energy and loudness are more concentrated around the middle of their ranges.

## 7. Box plots and outlier review

In [7]:
plt.figure(figsize=(8, 5))
sns.boxplot(x=df['tempo'])
plt.title('Tempo Distribution')
plt.xlabel('Tempo (BPM)')
plt.tight_layout()
plt.show()

q1 = df['tempo'].quantile(0.25)
q3 = df['tempo'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
outliers = df[(df['tempo'] < lower_bound) | (df['tempo'] > upper_bound)]
print('Q1:', q1)
print('Q3:', q3)
print('IQR:', iqr)
print('Lower bound:', lower_bound)
print('Upper bound:', upper_bound)
print('Number of tempo outliers:', len(outliers))
print(f'Tempo outlier percentage: {len(outliers)/len(df)*100:.2f}%')

Q1: 100.16399999999999
Q3: 137.69525
IQR: 37.53125
Lower bound: 43.86712499999999
Upper bound: 193.992125
Number of tempo outliers: 16
Tempo outlier percentage: 0.80%


/tmp/ipykernel_599619/124360366.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The IQR rule flagged 15 tempo values as potential outliers. They were kept because unusual tempos may still reflect real songs rather than data-entry mistakes.

## 8. Bivariate analysis

In [8]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='target', y='energy')
plt.title('Energy Distribution by Target Class')
plt.xlabel('Target Class')
plt.ylabel('Energy')
plt.tight_layout()
plt.show()

/tmp/ipykernel_599619/3449240907.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The box plot shows overlapping energy ranges between the classes, so energy alone does not clearly separate them.

In [9]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='energy', y='loudness', hue='target', alpha=0.65)
plt.title('Energy vs Loudness by Target Class')
plt.xlabel('Energy')
plt.ylabel('Loudness')
plt.tight_layout()
plt.show()

/tmp/ipykernel_599619/1867363223.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='danceability', y='valence', hue='target', alpha=0.65)
plt.title('Danceability vs Valence by Target Class')
plt.xlabel('Danceability')
plt.ylabel('Valence')
plt.tight_layout()
plt.show()

/tmp/ipykernel_599619/2426488427.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Correlation analysis

In [11]:
correlation_matrix = numeric_df.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

/tmp/ipykernel_599619/2790850820.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
upper_triangle = correlation_matrix.where(np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool))
strongest_pairs = upper_triangle.stack().sort_values(key=lambda values: values.abs(), ascending=False)
print('Strongest correlation pairs:')
print(strongest_pairs.head(5))

Strongest correlation pairs:
energy            loudness    0.762286
acousticness      energy     -0.647216
                  loudness   -0.561311
danceability      valence     0.442092
instrumentalness  loudness   -0.353337
dtype: float64


## 10. Pairplot review

In [13]:
selected_features = ['energy', 'loudness', 'danceability', 'valence', 'target']
sns.pairplot(df[selected_features], hue='target', corner=True, plot_kws={'alpha': 0.5})
plt.show()

/tmp/ipykernel_599619/1844937821.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Final Data-Storytelling Summary

The dataset contains music tracks with several numerical features and a target label. The data quality review found no missing values, and 5 duplicate rows were removed before the analysis. The distributions showed that some variables, such as tempo and danceability, spread across a broad range, while energy and loudness were more concentrated. The IQR method flagged 15 tempo outliers, and those values were retained because they may represent real songs with unusual tempo values rather than measurement errors. The strongest relationship in the dataset was the positive association between energy and loudness, with a correlation of about 0.76. Danceability and valence were also positively related but less strongly. These relationships may help with future modeling, especially if the model can use several correlated features, but the dataset should also be checked for redundancy and possible overfitting.

## 11. Modeling considerations

The class balance is close enough that the dataset can be used for a first modeling pass without major resampling. Energy, loudness, danceability, and valence appear to contain useful information, but some features may be redundant because several of them are correlated. The outlier review should be kept in mind during model training, especially if the model is sensitive to extreme values.